# Day 7：综合项目实践 — 带 RAG 的多轮对话 Agent> **目标**：将 Day1-Day6 的所有能力整合成一个可运行的知识库助手系统：ChatSessionManager 管理多轮对话，Agent 负责决策与工具调用，RAG 提供外部知识检索；完成端到端演示和系统级评估。---## 总体流程```textPart 1: 复用核心组件（精简版）Part 2: ChatSessionManager — 多轮对话管理器Part 3: 知识库构建与 RAG 检索器Part 4: RAGTool — 把 RAG 封装为 Agent 工具Part 5: KnowledgeAgentApp — 整合系统Part 6: 端到端多轮问答演示Part 7: 系统级评估Part 8: 总结与面试要点```

In [ ]:
from __future__ import annotationsfrom dataclasses import dataclass, fieldfrom typing import Any, Callable, Dict, List, Optional, Setfrom collections import Counterimport jsonimport mathimport re

## Part 1：复用核心组件将 Day3（Agent）和 Day6（RAG）的核心类重新定义在本 notebook 中，方便独立运行。

In [ ]:
# ──── Day3 核心组件 ────@dataclassclass Tool:    name: str    description: str    func: Callable[..., Any]    parameters: Dict[str, Any] = field(default_factory=dict)    def run(self, **kwargs) -> Any:        return self.func(**kwargs)    def to_schema(self) -> Dict[str, Any]:        return {"name": self.name, "description": self.description, "parameters": self.parameters}class ToolRegistry:    def __init__(self):        self.tools: Dict[str, Tool] = {}    def register(self, tool: Tool):        self.tools[tool.name] = tool    def get(self, name: str) -> Tool:        if name not in self.tools:            raise KeyError(f"未知工具: {name}")        return self.tools[name]    def list_names(self) -> List[str]:        return list(self.tools.keys())    def describe_text(self) -> str:        lines = []        for tool in self.tools.values():            params = tool.parameters.get("properties", {})            param_str = ", ".join(f'{k}: {v.get("type", "?")}' for k, v in params.items())            lines.append(f"- {tool.name}({param_str}): {tool.description}")        return "\n".join(lines)TYPE_MAP = {"string": str, "integer": int, "number": (int, float), "boolean": bool, "array": list, "object": dict}def validate_tool_args(schema: Dict[str, Any], arguments: Dict[str, Any]) -> tuple:    properties = schema.get("properties", {})    required = schema.get("required", [])    for req in required:        if req not in arguments:            return False, f"缺少必填参数: {req}"    for name, value in arguments.items():        if name not in properties:            return False, f"未知参数: {name}"        expected = properties[name].get("type")        if expected and expected in TYPE_MAP and not isinstance(value, TYPE_MAP[expected]):            return False, f"参数 {name} 类型错误"    return True, ""@dataclassclass ToolResult:    status: str    result: Any    error: Optional[str] = None    tool_name: str = ""    def to_observation(self) -> str:        if self.status == "ok":            return json.dumps(self.result, ensure_ascii=False, default=str)        return f"[工具错误] {self.error}"def safe_execute_tool(registry: ToolRegistry, tool_name: str, arguments: Dict[str, Any]) -> ToolResult:    try:        tool = registry.get(tool_name)    except KeyError as e:        return ToolResult(status="error", result=None, error=str(e), tool_name=tool_name)    valid, msg = validate_tool_args(tool.parameters, arguments)    if not valid:        return ToolResult(status="error", result=None, error=msg, tool_name=tool_name)    try:        result = tool.run(**arguments)        return ToolResult(status="ok", result=result, tool_name=tool_name)    except Exception as e:        return ToolResult(status="error", result=None, error=f"{type(e).__name__}: {e}", tool_name=tool_name)print("✅ Agent 核心组件加载完成")

In [ ]:
# ──── Day6 核心组件 ────VOCAB = [    "transformer", "注意力", "自注意力", "encoder", "decoder",    "gpt", "自回归", "预训练", "scaling",    "lora", "低秩", "微调", "冻结", "合并",    "qlora", "量化", "nf4", "显存",    "agent", "工具", "react", "memory",    "rag", "检索", "知识库", "召回", "重排",    "embedding", "向量", "双塔", "infonce", "对比学习",    "query", "chunk", "索引",]def tokenize_chinese(text: str) -> List[str]:    tokens = []    buf = ""    for ch in text:        if ch.isascii() and ch.isalnum():            buf += ch        else:            if buf:                tokens.append(buf.lower())                buf = ""            if ch.strip() and not ch.isascii():                tokens.append(ch)    if buf:        tokens.append(buf.lower())    return tokensdef cosine_similarity(a: List[float], b: List[float]) -> float:    dot = sum(x * y for x, y in zip(a, b))    na = math.sqrt(sum(x * x for x in a))    nb = math.sqrt(sum(x * x for x in b))    return dot / (na * nb) if na > 0 and nb > 0 else 0.0class TFIDFEmbedder:    def __init__(self, vocab: List[str]):        self.vocab = vocab        self.idf: List[float] = []        self.fitted = False    def fit(self, corpus: List[str]):        n = len(corpus)        df = [0.0] * len(self.vocab)        for doc in corpus:            doc_lower = doc.lower()            for i, token in enumerate(self.vocab):                if token.lower() in doc_lower:                    df[i] += 1        self.idf = [math.log(n / (1 + d)) for d in df]        self.fitted = True    def embed(self, text: str) -> List[float]:        text_lower = text.lower()        text_len = max(len(text_lower), 1)        return [text_lower.count(t.lower()) / text_len * self.idf[i] for i, t in enumerate(self.vocab)]class BM25Retriever:    def __init__(self, k1=1.5, b=0.75):        self.k1, self.b = k1, b        self.chunks, self.doc_tokens, self.avgdl, self.idf_map = [], [], 0.0, {}    def fit(self, chunks):        self.chunks = chunks        self.doc_tokens = [tokenize_chinese(c["text"]) for c in chunks]        n = len(chunks)        self.avgdl = sum(len(t) for t in self.doc_tokens) / max(n, 1)        df = {}        for tokens in self.doc_tokens:            for t in set(tokens):                df[t] = df.get(t, 0) + 1        self.idf_map = {t: math.log((n - d + 0.5) / (d + 0.5) + 1) for t, d in df.items()}    def search(self, query, top_k=5):        q_tokens = tokenize_chinese(query)        scores = []        for idx, doc_toks in enumerate(self.doc_tokens):            score, dl, tf_map = 0.0, len(doc_toks), Counter(doc_toks)            for qt in q_tokens:                if qt not in self.idf_map: continue                tf = tf_map.get(qt, 0)                score += self.idf_map[qt] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl))            scores.append((score, idx))        scores.sort(reverse=True)        return [{"score": round(s, 4), "chunk_id": self.chunks[i]["chunk_id"],                 "doc_id": self.chunks[i]["doc_id"], "text": self.chunks[i]["text"]}                for s, i in scores[:top_k] if s > 0]class DenseRetriever:    def __init__(self, embed_fn):        self.embed_fn, self.chunks, self.vectors = embed_fn, [], []    def fit(self, chunks):        self.chunks = chunks        self.vectors = [self.embed_fn(c["text"]) for c in chunks]    def search(self, query, top_k=5):        q_vec = self.embed_fn(query)        scored = [(cosine_similarity(q_vec, dv), i) for i, dv in enumerate(self.vectors)]        scored.sort(reverse=True)        return [{"score": round(s, 4), "chunk_id": self.chunks[i]["chunk_id"],                 "doc_id": self.chunks[i]["doc_id"], "text": self.chunks[i]["text"]}                for s, i in scored[:top_k] if s > 0]class HybridRetriever:    def __init__(self, dense, bm25, alpha=0.5):        self.dense, self.bm25, self.alpha = dense, bm25, alpha    def _norm(self, results):        if not results: return {}        scores = [r["score"] for r in results]        mn, mx = min(scores), max(scores)        rng = mx - mn if mx > mn else 1.0        return {r["chunk_id"]: (r["score"] - mn) / rng for r in results}    def search(self, query, top_k=5):        dr = self.dense.search(query, top_k=top_k*2)        br = self.bm25.search(query, top_k=top_k*2)        ds, bs = self._norm(dr), self._norm(br)        chunk_map = {r["chunk_id"]: r for r in dr + br}        combined = [(self.alpha * ds.get(cid, 0) + (1-self.alpha) * bs.get(cid, 0), cid)                     for cid in set(ds) | set(bs)]        combined.sort(reverse=True)        return [{"score": round(s, 4), **{k: chunk_map[cid][k] for k in ["chunk_id", "doc_id", "text"]}}                for s, cid in combined[:top_k]]print("✅ RAG 核心组件加载完成")

## Part 2：ChatSessionManager — 多轮对话管理器连接 Day1 的 history 管理和 token 预算控制。

In [ ]:
def estimate_tokens(text: str) -> int:    chinese = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')    return int(chinese * 1.5 + (len(text) - chinese) * 0.3)class ChatSessionManager:    """    多轮对话会话管理器。    职责：    1. 保存 system prompt（永不丢弃）    2. 保存对话 history    3. 按 token 预算做截断    4. 提供格式化的 messages 列表    """    def __init__(self, system_prompt: str, max_context_tokens: int = 3000):        self.system_prompt = system_prompt        self.max_context_tokens = max_context_tokens        self.history: List[Dict[str, str]] = []    def add_user(self, content: str):        self.history.append({"role": "user", "content": content})    def add_assistant(self, content: str):        self.history.append({"role": "assistant", "content": content})    def add_tool_trace(self, trace: str):        self.history.append({"role": "tool", "content": trace})    def get_messages(self) -> List[Dict[str, str]]:        """返回截断后的 messages，system prompt 始终保留。"""        sys_tokens = estimate_tokens(self.system_prompt)        budget = self.max_context_tokens - sys_tokens        selected = []        total = 0        for msg in reversed(self.history):            msg_tokens = estimate_tokens(msg["content"])            if total + msg_tokens > budget:                break            selected.insert(0, msg)            total += msg_tokens        return [{"role": "system", "content": self.system_prompt}] + selected    def get_recent_topics(self) -> List[str]:        """从最近消息中提取话题关键词。"""        topics = []        keywords = ["lora", "qlora", "rag", "react", "transformer", "agent", "gpt", "infonce", "embedding"]        for msg in reversed(self.history[-6:]):            content_lower = msg["content"].lower()            for kw in keywords:                if kw in content_lower and kw not in topics:                    topics.append(kw)        return topics    @property    def turn_count(self) -> int:        return sum(1 for m in self.history if m["role"] == "user")    def reset(self):        self.history.clear()session = ChatSessionManager(    system_prompt="你是课程知识库助手。请基于检索到的资料回答问题，如果资料不足请明确说明。")session.add_user("什么是 LoRA？")session.add_assistant("LoRA 是一种参数高效微调方法。")print(f"会话轮数: {session.turn_count}")print(f"Messages: {json.dumps(session.get_messages(), ensure_ascii=False, indent=2)}")

## Part 3：知识库构建与 RAG 索引

In [ ]:
documents = [    {"doc_id": "week2_transformer", "text": "Transformer 是一种基于自注意力机制的序列模型。自注意力通过 Query Key Value 计算注意力权重。多头注意力让模型捕捉不同模式。"},    {"doc_id": "week3_gpt", "text": "GPT 是基于 Decoder-only Transformer 的自回归语言模型。通过 next token prediction 预训练。GPT-3 证明了 scaling law。"},    {"doc_id": "week6_lora", "text": "LoRA 是参数高效微调方法。用低秩矩阵 BA 近似权重更新。只训练 A B 两个小矩阵，冻结预训练权重。推理时可合并回原权重。"},    {"doc_id": "week6_qlora", "text": "QLoRA 在 LoRA 基础上引入 NF4 量化。双重量化压缩量化常数。分页优化器缓解显存不足。让单卡微调 65B 模型成为可能。"},    {"doc_id": "week8_agent", "text": "Agent 是 LLM 加工具调用加状态管理的系统。ReAct 在 Thought Action Observation 间循环。Function Calling 用 JSON Schema 描述工具接口。"},    {"doc_id": "week8_rag", "text": "RAG 是检索增强生成。离线完成文档切分向量化和索引。在线完成 query 向量化 top-k 召回和增强生成。优化方向有混合检索重排和压缩。"},    {"doc_id": "week8_embedding", "text": "文本嵌入把语义相近句子映射到相近向量。双塔架构分别编码 query 和 doc。InfoNCE 是对比学习核心损失。交叉编码器适合重排。"},]def overlap_split(text, chunk_size=50, overlap=10):    chunks, start, step = [], 0, chunk_size - overlap    while start < len(text):        end = min(start + chunk_size, len(text))        chunks.append(text[start:end])        if end == len(text): break        start += step    return chunksall_chunks = []for doc in documents:    for idx, ct in enumerate(overlap_split(doc["text"], 50, 10)):        all_chunks.append({"chunk_id": f"{doc['doc_id']}_{idx}", "doc_id": doc["doc_id"], "text": ct})# 构建检索器tfidf = TFIDFEmbedder(VOCAB)tfidf.fit([c["text"] for c in all_chunks])dense_ret = DenseRetriever(tfidf.embed)dense_ret.fit(all_chunks)bm25_ret = BM25Retriever()bm25_ret.fit(all_chunks)hybrid_ret = HybridRetriever(dense_ret, bm25_ret, alpha=0.5)print(f"知识库: {len(documents)} 篇文档, {len(all_chunks)} 个 chunk")print("✅ RAG 索引构建完成")

## Part 4：RAGTool — 把 RAG 封装为 Agent 可调用工具

In [ ]:
def rag_search(query: str, top_k: int = 3) -> List[Dict[str, str]]:    """RAG 检索工具：返回与 query 最相关的知识片段。"""    results = hybrid_ret.search(query, top_k=top_k)    return [        {"source": r["doc_id"], "content": r["text"], "score": str(r["score"])}        for r in results    ]def calculator(expression: str) -> float:    allowed = {"abs": abs, "round": round, "sqrt": math.sqrt, "pow": pow}    return eval(expression, {"__builtins__": {}}, allowed)def get_current_time() -> str:    return "2026-03-06 14:30:00 (CST)"# 注册工具app_registry = ToolRegistry()app_registry.register(Tool(    name="rag_search",    description="检索课程知识库，返回相关知识片段和来源",    func=rag_search,    parameters={        "type": "object",        "properties": {            "query": {"type": "string", "description": "检索问题"},            "top_k": {"type": "integer", "description": "返回数量"},        },        "required": ["query"],    },))app_registry.register(Tool(    name="calculator",    description="执行数学计算",    func=calculator,    parameters={        "type": "object",        "properties": {"expression": {"type": "string", "description": "数学表达式"}},        "required": ["expression"],    },))app_registry.register(Tool(    name="get_current_time",    description="获取当前时间",    func=get_current_time,    parameters={"type": "object", "properties": {}},))print("已注册工具:", app_registry.list_names())print()# 测试 RAG 工具test_results = rag_search("LoRA 的原理", top_k=3)for r in test_results:    print(f"  [{r['source']}] {r['content'][:50]}...")

## Part 5：KnowledgeAgentApp — 整合系统将 ChatSessionManager + Agent + RAG 整合成一个完整的知识库助手应用。```text用户输入  ↓ChatSessionManager（保存 history、截断）  ↓Agent Planner（判断是否需要工具）  ↓ ← 如需检索 → RAGTool  ↓ ← 如需计算 → CalculatorLLM Generator（基于检索结果生成答案）  ↓写回 Session + 返回用户```

In [ ]:
def knowledge_planner(    user_query: str,    scratchpad: List[Dict[str, Any]],    tools: List[str],    session: ChatSessionManager = None,) -> Dict[str, Any]:    """    知识库助手的 planner。    决策逻辑：    1. 如果 scratchpad 中已有检索结果 -> 生成最终答案    2. 如果问题涉及课程内容 -> 调用 rag_search    3. 如果问题涉及计算 -> 调用 calculator    4. 否则 -> 直接回答    """    if scratchpad:        last = scratchpad[-1]        if last.get("tool_result_status") == "error":            return {                "type": "final_answer",                "content": f"工具调用失败：{last['observation']}。我尝试直接回答。",            }        obs = last["observation"]        if last["action"] == "rag_search":            try:                results = json.loads(obs)                if results:                    sources = set(r["source"] for r in results)                    content = " ".join(r["content"] for r in results)                    return {                        "type": "final_answer",                        "content": f"根据课程资料（来源: {', '.join(sources)}）：{content}",                    }            except (json.JSONDecodeError, KeyError):                pass            return {"type": "final_answer", "content": f"检索结果：{obs}"}        return {"type": "final_answer", "content": f"结果：{obs}"}    # 利用 session 做指代消解    resolved = user_query    if session:        topics = session.get_recent_topics()        if topics and any(w in user_query for w in ["它", "这个", "那个", "上面", "刚才", "前面"]):            resolved = user_query + f" ({topics[0]})"    query_lower = resolved.lower()    knowledge_keywords = [        "lora", "qlora", "rag", "react", "transformer", "gpt", "agent",        "embedding", "infonce", "注意力", "微调", "量化", "检索", "对比学习",        "什么是", "解释", "介绍", "原理", "区别", "为什么", "如何",    ]    if any(kw in query_lower for kw in knowledge_keywords):        return {            "type": "tool_call",            "name": "rag_search",            "arguments": {"query": resolved, "top_k": 3},            "thought": "这个问题涉及课程知识，先检索资料。",        }    if any(op in user_query for op in ["+", "-", "*", "/", "计算", "等于"]):        expr = user_query.replace("计算", "").replace("等于多少", "").strip()        return {            "type": "tool_call",            "name": "calculator",            "arguments": {"expression": expr},            "thought": "用户需要计算。",        }    if "时间" in user_query or "几点" in user_query:        return {"type": "tool_call", "name": "get_current_time", "arguments": {}, "thought": "查时间。"}    return {"type": "final_answer", "content": "这个问题不在课程知识库范围内，建议参考相关教材。"}

In [ ]:
class KnowledgeAgentApp:    """    带 RAG 的多轮对话 Agent 应用。    整合了：    - Day1: ChatSessionManager (多轮对话 + 截断)    - Day3: Agent Loop (工具调用 + 错误处理)    - Day6: RAG (知识库检索)    """    def __init__(        self,        planner: Callable,        registry: ToolRegistry,        system_prompt: str = "你是课程知识库助手。",        max_context_tokens: int = 3000,        max_steps: int = 4,        verbose: bool = True,    ):        self.planner = planner        self.registry = registry        self.session = ChatSessionManager(system_prompt, max_context_tokens)        self.max_steps = max_steps        self.verbose = verbose        self.stats = {"total_queries": 0, "tool_calls": 0, "tool_errors": 0}    def chat(self, user_input: str) -> str:        """处理一轮用户输入。"""        self.stats["total_queries"] += 1        if self.verbose:            print(f"\n{'='*60}")            print(f"[轮次 {self.session.turn_count + 1}] 用户: {user_input}")        self.session.add_user(user_input)        scratchpad: List[Dict[str, Any]] = []        for step in range(self.max_steps):            decision = self.planner(                user_input, scratchpad, self.registry.list_names(),                session=self.session,            )            if decision["type"] == "final_answer":                answer = decision["content"]                self.session.add_assistant(answer)                if self.verbose:                    print(f"助手: {answer[:120]}{'...' if len(answer) > 120 else ''}")                return answer            tool_name = decision["name"]            arguments = decision["arguments"]            if self.verbose:                print(f"  🔧 {tool_name}({json.dumps(arguments, ensure_ascii=False)[:60]})")            tool_result = safe_execute_tool(self.registry, tool_name, arguments)            self.stats["tool_calls"] += 1            if tool_result.status == "error":                self.stats["tool_errors"] += 1            self.session.add_tool_trace(                f"[{tool_name}] {tool_result.to_observation()[:100]}"            )            scratchpad.append({                "thought": decision.get("thought", ""),                "action": tool_name,                "arguments": arguments,                "observation": tool_result.to_observation(),                "tool_result_status": tool_result.status,            })        answer = "处理步骤已达上限。"        self.session.add_assistant(answer)        return answer    def show_stats(self):        print(f"\n--- 系统统计 ---")        print(f"总查询数: {self.stats['total_queries']}")        print(f"工具调用: {self.stats['tool_calls']}")        print(f"工具错误: {self.stats['tool_errors']}")        print(f"对话轮数: {self.session.turn_count}")        success_rate = (1 - self.stats['tool_errors'] / max(self.stats['tool_calls'], 1)) * 100        print(f"工具成功率: {success_rate:.1f}%")    def reset(self):        self.session.reset()        self.stats = {"total_queries": 0, "tool_calls": 0, "tool_errors": 0}app = KnowledgeAgentApp(    planner=knowledge_planner,    registry=app_registry,    system_prompt="你是课程知识库助手。请基于检索到的资料回答问题，如果资料不足请明确说明。",    verbose=True,)print("✅ KnowledgeAgentApp 初始化完成")

## Part 6：端到端多轮问答演示

In [ ]:
# 场景 1: 基础知识问答app.chat("请介绍一下 LoRA 的核心原理")# 场景 2: 指代消解 —— "它" 应该指 LoRAapp.chat("它和 QLoRA 有什么区别？")# 场景 3: 切换话题到 RAGapp.chat("RAG 的检索流程是怎样的？")# 场景 4: 跨领域对比app.chat("Agent 和 RAG 有什么关系？")# 场景 5: 非知识库问题app.chat("计算 256 * 32")# 场景 6: 时间查询app.chat("现在几点了？")

In [ ]:
app.show_stats()

## Part 7：系统级评估从四个维度评估综合系统：1. **检索质量**：top-k 是否命中正确文档2. **回答质量**：答案是否基于检索材料3. **多轮一致性**：是否能理解上文指代4. **工具可靠性**：工具调用成功率

In [ ]:
# 评估 1: 检索质量eval_queries = [    ("LoRA 的原理", {"week6_lora"}),    ("QLoRA 如何节省显存", {"week6_qlora"}),    ("RAG 的流程", {"week8_rag"}),    ("Transformer 自注意力", {"week2_transformer"}),    ("Agent 工具调用", {"week8_agent"}),]print("=" * 60)print("评估 1: 检索质量 (RAG 工具)")print("=" * 60)hits = 0for query, expected_docs in eval_queries:    results = rag_search(query, top_k=3)    found_docs = set(r["source"] for r in results)    hit = bool(expected_docs & found_docs)    hits += hit    status = "✅" if hit else "❌"    print(f"  {status} '{query}' -> 期望 {expected_docs}, 检索到 {found_docs}")print(f"\n检索命中率: {hits}/{len(eval_queries)} = {hits/len(eval_queries)*100:.0f}%")

In [ ]:
# 评估 2: 多轮一致性print("\n" + "=" * 60)print("评估 2: 多轮一致性")print("=" * 60)test_app = KnowledgeAgentApp(    planner=knowledge_planner, registry=app_registry, verbose=False,)r1 = test_app.chat("请介绍 LoRA")r2 = test_app.chat("它的核心公式是什么？")has_lora_ref = "lora" in r2.lower() or "低秩" in r2 or "week6" in r2.lower()print(f"  第一轮回答: {r1[:60]}...")print(f"  第二轮回答（'它'应指LoRA）: {r2[:60]}...")print(f"  指代消解: {'✅' if has_lora_ref else '⚠️ 可能未正确消解'}")

In [ ]:
# 评估 3: 工具可靠性print("\n" + "=" * 60)print("评估 3: 工具可靠性")print("=" * 60)test_app2 = KnowledgeAgentApp(    planner=knowledge_planner, registry=app_registry, verbose=False,)test_cases = [    ("什么是 Transformer？", "rag_search"),    ("计算 2 + 3", "calculator"),    ("现在几点", "get_current_time"),    ("你好", None),  # 不应调用工具]for query, expected_tool in test_cases:    test_app2.reset()    test_app2.chat(query)    actual_calls = test_app2.stats["tool_calls"]    actual_errors = test_app2.stats["tool_errors"]    if expected_tool is None:        status = "✅" if actual_calls == 0 else "⚠️"        print(f"  {status} '{query}' -> 期望无工具调用, 实际调用 {actual_calls} 次")    else:        status = "✅" if actual_calls > 0 and actual_errors == 0 else "❌"        print(f"  {status} '{query}' -> 期望 {expected_tool}, 调用 {actual_calls} 次, 错误 {actual_errors}")

In [ ]:
# 评估汇总print("\n" + "=" * 60)print("综合评估汇总")print("=" * 60)print("""| 维度         | 指标              | 结果        ||-------------|-------------------|------------|| 检索质量     | RAG top-3 命中率   | 见上方评估   || 回答质量     | 基于证据作答       | Mock 生成   || 多轮一致性   | 指代消解           | 见上方评估   || 工具可靠性   | 工具调用成功率     | 见上方评估   |当前系统是一个最小可行原型。真实系统需要：- 接入真实 LLM 替代规则化 planner 和 generator- 使用 Sentence-Transformers / BGE 做 embedding- 使用 FAISS / Chroma 做向量索引- 增加 reranker 提升精度- 构建标准化评测集""")

## Part 8：总结与面试要点### 8.1 第八周知识串联```textDay 1: 多轮对话 = history + 截断 + 模板        ↓Day 2-3: + 工具调用 = Agent (ReAct / Function Calling)        ↓Day 4-5: + 外部知识 = RAG (embedding / 检索 / 对比学习)        ↓Day 6: 手写 RAG 管线        ↓Day 7: 多轮对话 + Agent + RAG = 知识库助手```### 8.2 三层架构总结| 层 | 职责 | 核心类 ||----|------|--------|| 会话层 | history 管理、截断、模板 | ChatSessionManager || 决策层 | 是否调工具、调哪个、参数 | Agent (planner + loop) || 知识层 | 检索外部文档、提供证据 | RAG (retriever + reranker) |### 8.3 面试高频1. **画出 Agent + RAG 系统架构**：会话 -> 决策 -> 检索 -> 生成2. **Agent Loop 实现**：planner -> tool_call -> observation -> final_answer3. **RAG Pipeline 实现**：切分 -> embedding -> 检索 -> 重排 -> 生成4. **多轮对话 token 预算管理**5. **BM25 vs Dense 检索优缺点**### 8.4 后续方向1. 接入真实 LLM（OpenAI API / 本地模型）2. 使用 LangChain / LlamaIndex 封装3. 增加 reranker 和 query 改写4. 加入长期记忆（向量数据库）5. Web UI 或 API 服务化部署6. 构建评测集做系统化 benchmark---> **第八周完成！** 你已经从"会训练模型"走到了"会搭建 LLM 应用系统"。多轮对话是基础设施，Agent 提供决策与行动能力，RAG 提供外部知识访问能力。三者整合后，才是一个真正可用的知识库助手。